# SpendDNA Rahul Sharma's Wallet, Decoded
**Project:** The Unlox Academy — Week 2 Industry-Graded Minor Project
**Name:** Rishi Bhuta
**Dataset:** `Data_set_for_DADS_June.csv` (1,328 rows, Jan–Jun 2024)

Six months of a Bengaluru software engineer's UPI transactions, cleaned from scratch and turned into a
Spotify-Wrapped-style spending report — built with only Python fundamentals, NumPy and Pandas.
No regex, no ML, no matplotlib.

## Setup

In [15]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RAW_PATH = 'rahul_transactions.csv'  # place Data_set_for_DADS_June.csv here, renamed, or edit this path
try:
    raw = pd.read_csv(RAW_PATH)
except FileNotFoundError:
    RAW_PATH = 'Data_set_for_DADS_June.csv'
    raw = pd.read_csv(RAW_PATH)

print(f"Loaded {len(raw)} rows from {RAW_PATH}")
raw.head()

Loaded 1328 rows from rahul_transactions.csv


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


## Feature 1 — The Transaction Parser

The raw file mixes **4 date formats**, **3 amount formats**, and two spellings of Debit/Credit in a single
column. Everything downstream depends on getting this right, so it gets its own careful cell.

**Note on dates:** a plain `pd.to_datetime(..., dayfirst=True)` actually mis-parses this file — with four
formats mixed together, pandas' format inference silently swaps day/month on some of the unambiguous
ISO (`YYYY-MM-DD`) rows too. Instead, each date string is routed to the correct explicit `strftime` format
based on its shape (contains a space → `"12 Apr 2024"`; contains `/` → `"12/04/24"`; starts with a
4-digit year → ISO; otherwise → `"12-Apr-24"`). This is the safer, more surgical version of the same idea
the brief describes.

In [16]:
def parse_date(date_str):
    """Route each date string to its matching format based on its shape, then parse it."""
    s = str(date_str).strip()
    if ' ' in s:                                   # "12 Apr 2024"
        return pd.to_datetime(s, format='%d %b %Y', errors='coerce')
    elif '/' in s:                                  # "12/04/24"
        return pd.to_datetime(s, format='%d/%m/%y', errors='coerce')
    elif len(s) >= 5 and s[:4].isdigit() and s[4] == '-':   # "2024-04-12"
        return pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    else:                                            # "12-Apr-24"
        return pd.to_datetime(s, format='%d-%b-%y', errors='coerce')


def clean_amount(amount_series):
    """Strip currency symbols / commas from the three amount formats and convert to float."""
    cleaned = (amount_series.astype(str)
               .str.replace('₹', '', regex=False)
               .str.replace('Rs.', '', regex=False)
               .str.replace(',', '', regex=False)
               .str.strip())
    return pd.to_numeric(cleaned, errors='coerce')


# 1. Drop exact duplicate rows
rows_before = len(raw)
df = raw.drop_duplicates().copy()
dupes_dropped = rows_before - len(df)

# 2. Parse dates (all 4 formats)
df['date'] = df['Date'].apply(parse_date)

# 3. Parse amounts (all 3 formats)
df['amount'] = clean_amount(df['Amount'])

# 4. Standardise Type -> lowercase 'debit' / 'credit'
df['type_clean'] = df['Type'].str.lower().replace({'dr': 'debit', 'cr': 'credit'})

# 5. Treat empty Mode strings as missing
df['Mode'] = df['Mode'].replace('', np.nan)

# 6. Extract hour, month, day-of-week for later features
df['hour'] = df['Time'].str[:2].astype(int)
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()

unparsed_dates = df['date'].isna().sum()
unparsed_amounts = df['amount'].isna().sum()

# Drop any row that still failed to parse (should be none)
df = df.dropna(subset=['date', 'amount'])

print(f"Parsed {len(df)} transactions across 6 months. Dropped {dupes_dropped} duplicates. "
      f"{unparsed_dates} unparseable dates, {unparsed_amounts} unparseable amounts.")
print(f"\ndtypes check -> date: {df['date'].dtype}, amount: {df['amount'].dtype}")
print(f"shape: {df.shape}")

Parsed 1310 transactions across 6 months. Dropped 18 duplicates. 0 unparseable dates, 0 unparseable amounts.

dtypes check -> date: datetime64[ns], amount: float64
shape: (1310, 14)


## Feature 2 — Vendor Extractor

The `Description` column buries real merchant names inside UPI/POS/BHIM prefixes, bank-suffix tags,
and — trickiest of all — **parent-company legal names** that appear on real bank statements instead of
the consumer brand (e.g. `BUNDL Tech P L` is Swiggy's parent entity, `ANI Technologies` is Ola's).

Approach: inspect every unique description first, then build a `{canonical_vendor: [keywords]}`
dictionary and match with plain `.str.contains` / substring checks — **no regex**, as required.

Keyword order matters: more specific vendors (e.g. `Swiggy Instamart`) are checked *before* their
broader sibling (`Swiggy`), so `UPI-SWIGGY-INSTAMART@OKAXIS` doesn't get miscategorised.

In [17]:
# Inspect the raw variety first (this is what informed the dictionary below)
print(f"{df['Description'].nunique()} unique description strings in the file")
df['Description'].unique()[:15]

283 unique description strings in the file


array(['AMAZON SELLER SVCS', 'BHIM-BMTC',
       'NEFT-TECHCRUSH LABS-SALARY MAY24', 'UPI-AMAN-8934@OKAXIS',
       'BHIM-BLINKIT', 'BHIM ZEPTO', 'UPI-UBER-2426@HDFCBANK',
       'POS SWIGGY BANGALORE', 'UPI-GROWWPAY@HDFCBANK', 'OLA ELECTRIC',
       'BMS MOVIE TICKETS', 'POS OLA-PRIME', 'SWIGGY-INSTAMART',
       'UPI-STARBUCKS@AXIS', 'UPI-THIRDWAVE@OKAXIS'], dtype=object)

In [18]:
# canonical_vendor -> list of keywords (all matched case-insensitively via substring search)
vendor_keywords = {
    'Swiggy Instamart':      ['SWIGGY-INSTAMART', 'SWIGGY INSTAMART', 'BUNDL TECH-INSTAMART', 'INSTAMART'],
    'Zomato':                ['ZOMATO'],
    'Swiggy':                ['SWIGGY', 'BUNDL'],                       # BUNDL Tech P L = Swiggy's parent co.
    'Blinkit':                ['BLINKIT', 'GROFERS'],                    # Grofers = Blinkit's old/parent name
    'Zepto':                  ['ZEPTO', 'KIRANAKART'],                   # Kiranakart = Zepto's parent co.
    'BigBasket':               ['BIGBASKET'],
    'DMart':                   ['DMART', 'AVENUE SUPERMARTS'],
    'Amazon Prime':            ['AMAZON PRIME', 'AMZN PRIME', 'AMAZON-PRIME'],
    'Amazon':                  ['AMAZON', 'AMZN'],
    'Flipkart':                ['FLIPKART', 'FKART', 'INNOVATIVE RETAIL'],
    'Myntra':                  ['MYNTRA'],
    'Nykaa':                   ['NYKAA', 'FSN E-COMMERCE'],              # FSN E-Commerce = Nykaa's legal name
    'Uber':                    ['UBER'],
    'Ola':                     ['OLA', 'ANI TECHNOLOGIES', 'ROPPEN TRANSPORTATION'],  # Ola's legal entities
    'Rapido':                  ['RAPIDO'],
    'BMTC':                    ['BMTC', 'TUMMOC'],
    'Starbucks':               ['STARBUCKS'],
    'Third Wave Coffee':       ['THIRDWAVE', 'THIRD WAVE COFFEE', 'TWC'],
    'Cafe Coffee Day':         ['COFFEE DAY', 'CCD'],
    'BookMyShow':              ['BOOKMYSHOW', 'BMS MOVIE', 'BIGTREE'],   # Bigtree Entertainment = BMS parent
    'Netflix':                 ['NETFLIX'],
    'Spotify':                 ['SPOTIFY'],
    'Disney+ Hotstar':         ['HOTSTAR', 'STAR INDIA'],
    'JioFiber':                ['JIOFIBER'],
    'Jio':                     ['JIORECHARGE', 'RELIANCE JIO'],
    'Airtel':                  ['AIRTEL', 'BHARTI AIRTEL'],
    'Vi':                      ['VODAFONE', 'VI-RECHARGE', 'VI POSTPAID'],
    'BESCOM':                  ['BESCOM', 'BANGALORE ELEC'],
    'BWSSB':                   ['BWSSB'],
    'Zerodha':                 ['ZERODHA'],
    'Groww':                   ['GROWW', 'NEXTBILLION'],                 # Nextbillion = Groww's legal entity
    'BPCL':                    ['BPCL'],
    'HP Petrol':               ['HP PETROL'],
    'Indian Oil':              ['INDIAN OIL', 'IOC'],
    'Landlord (Rent)':         ['RENT-LANDLORD'],
    'Salary':                  ['SALARY'],
    'Restaurant (Dine-in)':    ['RESTAURANT', 'TRUFFLES', 'MEGHANA FOODS', 'EMPIRE RESTAURANT', 'DINEOUT', 'DINING'],
}

# Explicit priority order: specific vendors checked before their broader siblings
vendor_order = ['Swiggy Instamart', 'Zomato', 'Swiggy', 'Blinkit', 'Zepto', 'BigBasket', 'DMart',
                 'Amazon Prime', 'Amazon', 'Flipkart', 'Myntra', 'Nykaa',
                 'Uber', 'Ola', 'Rapido', 'BMTC',
                 'Starbucks', 'Third Wave Coffee', 'Cafe Coffee Day',
                 'BookMyShow', 'Netflix', 'Spotify', 'Disney+ Hotstar', 'JioFiber', 'Jio', 'Airtel', 'Vi',
                 'BESCOM', 'BWSSB', 'Zerodha', 'Groww', 'BPCL', 'HP Petrol', 'Indian Oil',
                 'Landlord (Rent)', 'Salary', 'Restaurant (Dine-in)']


def extract_vendor(description):
    """Match a messy description string against the vendor dictionary; handle P2P and ATM specially."""
    text = description.upper()
    for vendor in vendor_order:
        for keyword in vendor_keywords[vendor]:
            if keyword in text:
                return vendor
    if text.startswith('ATM-WDL'):
        return 'Cash Withdrawal'
    if text.startswith('UPI-') and '@' in text:      # friend-to-friend handles, e.g. UPI-PRIYA-9876@OKAXIS
        return 'P2P Transfer'
    return 'Uncategorised'


df['vendor_clean'] = df['Description'].apply(extract_vendor)

print(f"df['vendor_clean'].nunique() = {df['vendor_clean'].nunique()} canonical vendors")
print("\nTop 10 vendors by transaction count:")
print(df['vendor_clean'].value_counts().head(10))

df['vendor_clean'].nunique() = 39 canonical vendors

Top 10 vendors by transaction count:
vendor_clean
Swiggy                  176
Zomato                  121
Ola                     101
Amazon                   76
Restaurant (Dine-in)     73
Zepto                    71
Uber                     71
Swiggy Instamart         67
Flipkart                 55
Blinkit                  55
Name: count, dtype: int64


## Feature 3 — Category Tagger

Every canonical vendor is mapped to one of the 12 spending categories, plus three non-consumption
buckets that get tracked separately: **Personal Transfer**, **Cash Withdrawal**, and **Rent** (a fixed
monthly outgoing, not a discretionary "spend" category — same logic the brief applies to P2P transfers).
Salary is tagged as `Income` since it's a credit, not a debit.

In [19]:
category_map = {
    'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Swiggy Instamart': 'Quick Commerce', 'Blinkit': 'Quick Commerce', 'Zepto': 'Quick Commerce',
    'BigBasket': 'Groceries', 'DMart': 'Groceries',
    'Amazon': 'Ecommerce', 'Amazon Prime': 'Subscriptions', 'Flipkart': 'Ecommerce',
    'Myntra': 'Ecommerce', 'Nykaa': 'Ecommerce',
    'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'BMTC': 'Transport',
    'Starbucks': 'Cafe', 'Third Wave Coffee': 'Cafe', 'Cafe Coffee Day': 'Cafe',
    'Restaurant (Dine-in)': 'Restaurants',
    'Netflix': 'Subscriptions', 'Spotify': 'Subscriptions',
    'Disney+ Hotstar': 'Subscriptions', 'JioFiber': 'Subscriptions',
    'Airtel': 'Utilities', 'Vi': 'Utilities', 'Jio': 'Utilities', 'BESCOM': 'Utilities', 'BWSSB': 'Utilities',
    'Zerodha': 'Investments', 'Groww': 'Investments',
    'BPCL': 'Fuel', 'HP Petrol': 'Fuel', 'Indian Oil': 'Fuel',
    'BookMyShow': 'Entertainment',
    'Landlord (Rent)': 'Rent', 'Salary': 'Income',
    'P2P Transfer': 'Personal Transfer', 'Cash Withdrawal': 'Cash Withdrawal',
}

df['category'] = df['vendor_clean'].map(category_map)

print("Transaction count by category:")
print(df['category'].value_counts())
print(f"\nUncategorised rows: {(df['category'].isna()).sum()}")

Transaction count by category:
category
Food Delivery        297
Transport            250
Quick Commerce       193
Ecommerce            170
Cafe                  99
Restaurants           73
Subscriptions         42
Utilities             42
Groceries             33
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64

Uncategorised rows: 0


## Feature 4 — Spending Overview

The executive summary: total credits/debits, net position, savings rate, and the top categories and
vendors by spend.

**Design choice:** Personal Transfers, Cash Withdrawals, and Rent are excluded from the *category
percentage-of-spend* table (they aren't discretionary "spend" the way a Swiggy order is), but they are
still counted in `total debits`. This mirrors the brief's own recommendation for P2P transfers.

In [20]:
debits_df = df[df['type_clean'] == 'debit']
credits_df = df[df['type_clean'] == 'credit']

total_credits = credits_df['amount'].sum()
total_debits = debits_df['amount'].sum()
net_change = total_credits - total_debits
savings_rate = net_change / total_credits * 100

NON_SPEND_CATEGORIES = ['Personal Transfer', 'Cash Withdrawal', 'Rent']
spend_df = debits_df[~debits_df['category'].isin(NON_SPEND_CATEGORIES)].copy()

category_totals = spend_df.groupby('category')['amount'].sum().sort_values(ascending=False)
category_pct = category_totals / spend_df['amount'].sum() * 100

vendor_totals = spend_df.groupby('vendor_clean').agg(total=('amount', 'sum'), orders=('amount', 'count'))
vendor_totals = vendor_totals.sort_values('total', ascending=False)

print(f"Total credits     : Rs. {total_credits:,.0f}")
print(f"Total debits       : Rs. {total_debits:,.0f}")
print(f"Net change         : Rs. {net_change:,.0f}")
print(f"Savings rate       : {savings_rate:.1f}%")
print(f"Transactions       : {len(df)}")
print(f"Unique vendors     : {df['vendor_clean'].nunique()}")

print("\nTop 5 categories by spend (% of core spend):")
print(category_pct.head(5).round(1))

print("\nTop 5 vendors by spend:")
print(vendor_totals.head(5))

Total credits     : Rs. 509,774
Total debits       : Rs. 1,678,901
Net change         : Rs. -1,169,127
Savings rate       : -229.3%
Transactions       : 1310
Unique vendors     : 39

Top 5 categories by spend (% of core spend):
category
Ecommerce         40.2
Investments       16.5
Food Delivery      8.6
Restaurants        7.8
Quick Commerce     6.4
Name: amount, dtype: float64

Top 5 vendors by spend:
                         total  orders
vendor_clean                          
Amazon                318422.0      76
Zerodha               210000.0      14
Flipkart              186709.0      55
Restaurant (Dine-in)  117737.0      73
Swiggy                 73738.0     176


## Feature 5 — Monthly Trend Analysis

A category × month pivot table, plus which category grew fastest and which declined fastest from
January to June (using plain NumPy percentage-change arithmetic, no `.pct_change()` shortcuts).

In [21]:
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}

month_pivot = spend_df.pivot_table(values='amount', index='category', columns='month',
                                    aggfunc='sum', fill_value=0)
month_pivot = month_pivot.rename(columns=month_names)[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']]

jan_vals = month_pivot['Jan'].to_numpy()
jun_vals = month_pivot['Jun'].to_numpy()
pct_change = np.where(jan_vals != 0, (jun_vals - jan_vals) / jan_vals * 100, np.nan)
growth = pd.Series(pct_change, index=month_pivot.index).sort_values(ascending=False)

print("Category x Month spend matrix (Rs.):")
print(month_pivot.round(0))

print("\nBiggest Jan -> Jun growth:")
print(growth.head(3).round(1))
print("\nBiggest Jan -> Jun decline:")
print(growth.tail(3).round(1))

Category x Month spend matrix (Rs.):
month               Jan      Feb       Mar      Apr      May       Jun
category                                                              
Cafe             3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Ecommerce       97134.0  93827.0  104538.0  73621.0  96857.0  136991.0
Entertainment    1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery   20890.0  21452.0   20850.0  23054.0  22167.0   20641.0
Fuel            30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries       17649.0   7517.0    5523.0   6225.0   7862.0    5432.0
Investments     38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce  12797.0  17465.0   17979.0  16572.0  15188.0   15666.0
Restaurants     16320.0  21772.0   28313.0   7711.0  22286.0   21335.0
Subscriptions    4256.0   5866.0    8517.0   3289.0   2619.0    4597.0
Transport       11005.0  10191.0    6857.0   9284.0  13141.0    6996.0
Utilities        9157.0   6870.0    4932

## Feature 6 — Time-of-Day Patterns

A category × hour-of-day matrix built with NumPy, used to surface behavioural insights like when food
delivery and cafe spending peak.

In [22]:
categories_sorted = sorted(spend_df['category'].unique())
hour_matrix = np.zeros((len(categories_sorted), 24), dtype=int)

for i, cat in enumerate(categories_sorted):
    hour_counts = spend_df[spend_df['category'] == cat]['hour'].value_counts()
    for hour, count in hour_counts.items():
        hour_matrix[i, hour] = count

hour_df = pd.DataFrame(hour_matrix, index=categories_sorted, columns=range(24))

# Food Delivery peak window
fd_hours = spend_df[spend_df['category'] == 'Food Delivery']['hour']
fd_night_pct = ((fd_hours >= 21) | (fd_hours < 2)).sum() / len(fd_hours) * 100
fd_peak_hour = fd_hours.value_counts().idxmax()

# Cafe peak window
cafe_hours = spend_df[spend_df['category'] == 'Cafe']['hour']
cafe_morning_pct = ((cafe_hours >= 8) & (cafe_hours <= 11)).sum() / len(cafe_hours) * 100
cafe_peak_hour = cafe_hours.value_counts().idxmax()

print(f"Food Delivery: busiest hour is {fd_peak_hour}:00, "
      f"{fd_night_pct:.1f}% of orders fall in the 21:00-02:00 late-night window")
print(f"Cafe: busiest hour is {cafe_peak_hour}:00, "
      f"{cafe_morning_pct:.1f}% of visits fall in the 08:00-11:00 morning window")

print("\nFood Delivery order count by hour:")
print(hour_df.loc['Food Delivery'][hour_df.loc['Food Delivery'] > 0])

Food Delivery: busiest hour is 20:00, 20.5% of orders fall in the 21:00-02:00 late-night window
Cafe: busiest hour is 10:00, 35.4% of visits fall in the 08:00-11:00 morning window

Food Delivery order count by hour:
0      1
1      7
2      2
3      4
4      6
5      7
6      1
7      2
8      8
9     10
10     6
11    18
12    20
13    12
14     9
15    15
16     9
17    10
18    27
19    34
20    36
21    22
22    22
23     9
Name: Food Delivery, dtype: int64


## Feature 7 — Anomaly Detection (Z-score)

For each category, compute the mean and standard deviation of transaction amounts, then flag any
transaction more than 2 standard deviations above its own category's mean. Note the z-score is computed
**within category** (via `groupby().transform()`), not across the whole dataset — a ₹2,000 Swiggy order
would be a huge outlier for Food Delivery but unremarkable for Ecommerce.

In [23]:
category_mean = spend_df.groupby('category')['amount'].transform('mean')
category_std = spend_df.groupby('category')['amount'].transform('std')
spend_df['z_score'] = (spend_df['amount'] - category_mean) / category_std

anomalies = spend_df[spend_df['z_score'] > 2].sort_values('z_score', ascending=False)

print(f"{len(anomalies)} anomalous transactions flagged (z-score > 2, i.e. top ~2% of their category)")
print("\nTop 5 anomalies:")
print(anomalies[['date', 'vendor_clean', 'category', 'amount', 'z_score']].head(5).to_string(index=False))

22 anomalous transactions flagged (z-score > 2, i.e. top ~2% of their category)

Top 5 anomalies:
      date         vendor_clean    category  amount  z_score
2024-06-26               Amazon   Ecommerce 22008.0 4.068661
2024-02-07               Amazon   Ecommerce 21986.0 4.063813
2024-02-26 Restaurant (Dine-in) Restaurants  8383.0 3.884639
2024-06-22 Restaurant (Dine-in) Restaurants  7935.0 3.627582
2024-03-31 Restaurant (Dine-in) Restaurants  7931.0 3.625287


## Feature 8 — Spending Archetype Detection

Each of the 8 archetype rules from the brief is implemented as its own function and applied to Rahul's
actual computed metrics. A transaction history can match multiple archetypes — every rule that fires
gets reported, along with the metric that triggered it.

**A note on the numbers:** this dataset is a distinct random generation from the brief's illustrative
example, so its category mix differs somewhat (heavier Ecommerce spend from a handful of large
anomalous purchases, lighter late-night food ordering). The rules below are applied faithfully to
*this* file's real numbers rather than forced to match the brief's sample output.

In [24]:
foodie_pct = category_pct.get('Food Delivery', 0) + category_pct.get('Restaurants', 0) + category_pct.get('Cafe', 0)
quick_commerce_pct = category_pct.get('Quick Commerce', 0)
ecommerce_pct = category_pct.get('Ecommerce', 0)
investments_pct = category_pct.get('Investments', 0)
transport_pct = category_pct.get('Transport', 0)
subscription_vendor_count = spend_df[spend_df['category'] == 'Subscriptions']['vendor_clean'].nunique()


def is_foodie(pct):
    return pct > 25, f"{pct:.1f}% on Food Delivery + Restaurants + Cafe combined"

def is_quick_commerce_junkie(pct):
    return pct > 15, f"{pct:.1f}% on Quick Commerce"

def is_shopaholic(pct):
    return pct > 15, f"{pct:.1f}% on Ecommerce"

def is_investor(pct):
    return pct > 15, f"{pct:.1f}% on Investments"

def is_late_night_snacker(pct):
    return pct > 50, f"{pct:.1f}% of Food Delivery orders between 21:00-02:00"

def is_cab_commuter(pct):
    return pct > 10, f"{pct:.1f}% on Transport"

def is_subscription_lover(count):
    return count >= 5, f"{count} distinct active subscription vendors"

def is_yolo_spender(rate):
    return rate < 10, f"savings rate is {rate:.1f}%"

def is_disciplined_saver(rate):
    return rate > 40, f"savings rate is {rate:.1f}%"


archetype_checks = [
    ('THE FOODIE', is_foodie(foodie_pct)),
    ('THE QUICK COMMERCE JUNKIE', is_quick_commerce_junkie(quick_commerce_pct)),
    ('THE SHOPAHOLIC', is_shopaholic(ecommerce_pct)),
    ('THE INVESTOR', is_investor(investments_pct)),
    ('THE LATE-NIGHT SNACKER', is_late_night_snacker(fd_night_pct)),
    ('THE CAB COMMUTER', is_cab_commuter(transport_pct)),
    ('THE SUBSCRIPTION LOVER', is_subscription_lover(subscription_vendor_count)),
    ('THE YOLO SPENDER', is_yolo_spender(savings_rate)),
    ('THE DISCIPLINED SAVER', is_disciplined_saver(savings_rate)),
]

detected_archetypes = [(name, detail) for name, (matched, detail) in archetype_checks if matched]

print("Archetype results:")
for name, (matched, detail) in archetype_checks:
    flag = 'MATCH' if matched else '  -  '
    print(f"  [{flag}] {name:<28} ({detail})")

print(f"\n-> Rahul matches {len(detected_archetypes)} archetype(s): "
      f"{', '.join(n for n, _ in detected_archetypes)}")

Archetype results:
  [  -  ] THE FOODIE                   (18.5% on Food Delivery + Restaurants + Cafe combined)
  [  -  ] THE QUICK COMMERCE JUNKIE    (6.4% on Quick Commerce)
  [MATCH] THE SHOPAHOLIC               (40.2% on Ecommerce)
  [MATCH] THE INVESTOR                 (16.5% on Investments)
  [  -  ] THE LATE-NIGHT SNACKER       (20.5% of Food Delivery orders between 21:00-02:00)
  [  -  ] THE CAB COMMUTER             (3.8% on Transport)
  [MATCH] THE SUBSCRIPTION LOVER       (5 distinct active subscription vendors)
  [MATCH] THE YOLO SPENDER             (savings rate is -229.3%)
  [  -  ] THE DISCIPLINED SAVER        (savings rate is -229.3%)

-> Rahul matches 4 archetype(s): THE SHOPAHOLIC, THE INVESTOR, THE SUBSCRIPTION LOVER, THE YOLO SPENDER


### Bonus — Invented Archetype: THE BENGALURU TRAFFIC SURVIVOR

Detection rule: **200+ Transport transactions in 6 months** (roughly one ride every day) — a pattern
specific to Bengaluru's traffic-heavy commute culture, where tech workers lean on Uber/Ola/Rapido/BMTC
constantly for short hops rather than owning or driving a car, even though the *rupee* spend on Transport
stays low (short rides are cheap; it's the sheer frequency that's Bengaluru-specific).

In [25]:
transport_txn_count = len(spend_df[spend_df['category'] == 'Transport'])
is_traffic_survivor = transport_txn_count >= 200

print(f"Transport transactions in 6 months: {transport_txn_count}")
print(f"THE BENGALURU TRAFFIC SURVIVOR: {'MATCH' if is_traffic_survivor else 'no match'} "
      f"({transport_txn_count} rides across Uber/Ola/Rapido/BMTC, "
      f"~{transport_txn_count/26:.1f} rides/week)")

if is_traffic_survivor:
    detected_archetypes.append(('THE BENGALURU TRAFFIC SURVIVOR',
                                 f"{transport_txn_count} rides in 6 months"))

Transport transactions in 6 months: 250
THE BENGALURU TRAFFIC SURVIVOR: MATCH (250 rides across Uber/Ola/Rapido/BMTC, ~9.6 rides/week)


## Bonus — Vendor Cleanup Audit

A well-built vendor extractor should have close to zero `Uncategorised` rows. Listing them here shows
the dictionary is complete for this dataset.

In [26]:
uncategorised = df[df['vendor_clean'] == 'Uncategorised']
print(f"Uncategorised rows: {len(uncategorised)}")
if len(uncategorised) > 0:
    print(uncategorised['Description'].unique())
else:
    print("None — every description in the file matched a vendor keyword.")

Uncategorised rows: 0
None — every description in the file matched a vendor keyword.


## Bonus — Day-of-Week Analysis

Extending Feature 6: is Rahul's spending meaningfully higher on weekends than weekdays?

In [27]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_totals = spend_df.groupby('day_of_week')['amount'].sum().reindex(dow_order)

weekday_total = dow_totals[['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']].sum()
weekend_total = dow_totals[['Saturday', 'Sunday']].sum()
weekday_avg_per_day = weekday_total / 5
weekend_avg_per_day = weekend_total / 2
weekend_premium_pct = (weekend_avg_per_day - weekday_avg_per_day) / weekday_avg_per_day * 100

print("Spend by day of week:")
print(dow_totals.round(0))
print(f"\nAvg weekday spend/day: Rs. {weekday_avg_per_day:,.0f}")
print(f"Avg weekend spend/day: Rs. {weekend_avg_per_day:,.0f}")
print(f"Weekend spending is {weekend_premium_pct:+.1f}% vs weekday average")

Spend by day of week:
day_of_week
Monday       207122.0
Tuesday      245368.0
Wednesday    292440.0
Thursday     183076.0
Friday       146969.0
Saturday     247243.0
Sunday       178584.0
Name: amount, dtype: float64

Avg weekday spend/day: Rs. 214,995
Avg weekend spend/day: Rs. 212,914
Weekend spending is -1.0% vs weekday average


## The Final Report

Everything above, assembled into one screenshot-worthy printed report.

In [28]:
def bar(pct, width=18):
    filled = int(round(pct / 100 * width))
    return '#' * filled + ' ' * (width - filled)


W = 64
print('=' * W)
print(' SpendDNA REPORT - RAHUL SHARMA')
print(f' 6 months - {len(df):,} transactions - Jan to Jun 2024')
print('=' * W)

print('\n EXECUTIVE SUMMARY')
print(f' Total credits   : Rs. {total_credits:,.0f}')
print(f' Total debits    : Rs. {total_debits:,.0f}')
status = 'overspending' if net_change < 0 else 'net saver'
print(f' Net change      : {"-" if net_change < 0 else ""}Rs. {abs(net_change):,.0f} ({status})')
savings_flag = 'BURNING SAVINGS' if savings_rate < 0 else ('HEALTHY' if savings_rate > 20 else 'TIGHT')
print(f' Savings rate    : {savings_rate:.1f}% ({savings_flag})')
print(f' Transactions    : {len(df):,}')
print(f' Unique vendors  : {df["vendor_clean"].nunique()}')

print('\n TOP CATEGORIES (% of core spend)')
for cat, pct in category_pct.head(5).items():
    amt = category_totals[cat]
    print(f' {cat:<16} {bar(pct)} {pct:5.1f}%  Rs. {amt:>10,.0f}')

print('\n TOP VENDORS')
for vendor, row in vendor_totals.head(5).iterrows():
    print(f' {vendor:<16} Rs. {row["total"]:>10,.0f}  ({int(row["orders"])} orders)')

print('\n TIME-OF-DAY PATTERNS')
print(f' Food Delivery peaks: {fd_peak_hour:02d}:00  ({fd_night_pct:.0f}% of orders in the 21:00-02:00 window)')
print(f' Cafe peaks:          {cafe_peak_hour:02d}:00  ({cafe_morning_pct:.0f}% of visits in the 08:00-11:00 window)')

print('\n MONTHLY TREND (Food Delivery)')
fd_row = month_pivot.loc['Food Delivery']
fd_max = fd_row.max()
for m, amt in fd_row.items():
    print(f' {m}  Rs. {amt:>8,.0f}  {bar(amt / fd_max * 100, 14)}')

print('\n TOP ANOMALIES (z-score > 2 within category)')
for _, row in anomalies.head(5).iterrows():
    print(f' {row["date"].strftime("%d %b")} - {row["vendor_clean"]:<12} Rs. {row["amount"]:>8,.0f}  (z={row["z_score"]:.1f})')

print('\n RAHUL\'S SPENDING ARCHETYPES')
for name, detail in detected_archetypes:
    print(f' -> {name}  ({detail})')

print('=' * W)

 SpendDNA REPORT - RAHUL SHARMA
 6 months - 1,310 transactions - Jan to Jun 2024

 EXECUTIVE SUMMARY
 Total credits   : Rs. 509,774
 Total debits    : Rs. 1,678,901
 Net change      : -Rs. 1,169,127 (overspending)
 Savings rate    : -229.3% (BURNING SAVINGS)
 Transactions    : 1,310
 Unique vendors  : 39

 TOP CATEGORIES (% of core spend)
 Ecommerce        #######             40.2%  Rs.    602,968
 Investments      ###                 16.5%  Rs.    248,160
 Food Delivery    ##                   8.6%  Rs.    129,054
 Restaurants      #                    7.8%  Rs.    117,737
 Quick Commerce   #                    6.4%  Rs.     95,667

 TOP VENDORS
 Amazon           Rs.    318,422  (76 orders)
 Zerodha          Rs.    210,000  (14 orders)
 Flipkart         Rs.    186,709  (55 orders)
 Restaurant (Dine-in) Rs.    117,737  (73 orders)
 Swiggy           Rs.     73,738  (176 orders)

 TIME-OF-DAY PATTERNS
 Food Delivery peaks: 20:00  (21% of orders in the 21:00-02:00 window)
 Cafe peaks:    

## Key Insights

1. **Ecommerce, not food delivery, is the real budget-breaker.** Ecommerce accounts for
   ~{ecom:.0f}% of core spend — well above Investments (~{inv:.0f}%) and Food Delivery (~{fd:.0f}%) —
   driven by a handful of large orders (several individual Amazon/Flipkart purchases exceed
   Rs. 15,000, each landing as a z-score anomaly). A few big-ticket months are doing more damage
   to the budget than the daily Swiggy/Zomato habit.

2. **The savings rate is deeply negative** (~{sr:.0f}%): Rahul is spending far more than his
   salary credits each month. Even with a healthy Zerodha SIP running, the discretionary side of
   the ledger (Ecommerce + Food Delivery + Quick Commerce + Restaurants + Cafe combined) is
   outpacing income by a wide margin — this is the number a fintech "Wrapped" report would lead
   with in red.

3. **He's a high-frequency, low-ticket commuter.** {transport_count} separate Transport
   transactions in 6 months (Uber, Ola, Rapido, BMTC) — almost one ride every single day — yet
   Transport is only a small slice of total spend by rupee value. Classic Bengaluru pattern: it's
   not that any one ride is expensive, it's that there's always one being booked.

*(Numbers above are computed live in the cells; this markdown mirrors the values already printed
in the report — see the Executive Summary and Top Categories sections above for the exact figures.)*

## Reflection

Building the parser was the easy part — the real work was the vendor dictionary. Real bank statements
don't say "Swiggy," they say `BUNDL Tech P L`, because that's the legal entity that actually processed
the payment. Getting from raw description to canonical brand name meant treating this like actual
fintech merchant-normalisation work: inspect every unique string, notice the parent-company aliases,
and build the dictionary keyword by keyword rather than guessing a generic pattern.

The archetype numbers on this specific file don't match the brief's illustrative example exactly
(Ecommerce dominates here instead of Food Delivery), which was a useful reminder that "does my code
run correctly" and "does my output match a sample screenshot" are two different questions — the second
one only matters if you're working with the exact same data.

**AI-assistance disclosure:** Claude (Anthropic) was used to help design the vendor-keyword dictionary,
debug the multi-format date parser, and structure the final report formatting. All category thresholds,
archetype logic, and analysis were verified by running against this specific dataset's actual numbers.